# RH White Glove Delivery Optimization

**Dataset 10 — 19 customers, 168 order lines**  
**Warehouse:** 590 20th St, San Francisco, CA 94107

---

## What this notebook does

| Step | Description |
|------|-------------|
| 1 | Install dependencies and import libraries |
| 2 | Load all three source files |
| 3 | DIMS dictionary — exact product dimensions from rh.com + Exhibit V times |
| 4 | Match customer addresses from the RH address database |
| 5 | Compute per-customer volume, weight, load time, and service time |
| 6 | Build the distance matrix (Google Maps API — hardcoded after first pull) |
| 7 | Assign customers to daily loads respecting truck capacity and time window |
| 8 | Objective function and constraints |
| 9 | Optimize each day's stop sequence with OR-Tools |
| 10 | Calculate costs (truck, labor with RH pay rounding, holding) and CO₂ |
| 11 | Export schedule, manifest, daily summary, and assumptions to Excel for Power BI |

**Author:** Coovadia Nyoka — Summer 2026 (team case project; model and code by author)

## Section 1 — Installing Dependencies

This cell installs OR-Tools (route optimizer), googlemaps (distance API), and openpyxl (Excel export).  
Run once per Colab session.

In [ ]:
!pip install ortools googlemaps openpyxl --quiet
print("All dependencies installed.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
All dependencies installed.


## Section 2 — Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
import re
from google.colab import files
import googlemaps

warnings.filterwarnings("ignore")

# OR-Tools — vehicle routing optimizer
try:
    from ortools.constraint_solver import routing_enums_pb2, pywrapcp
    ORTOOLS_AVAILABLE = True
    print("OR-Tools loaded successfully.")
except ImportError:
    ORTOOLS_AVAILABLE = False
    print("WARNING: OR-Tools not available. Will use nearest-neighbour fallback.")

# Google Maps
try:
    import googlemaps
    GMAPS_AVAILABLE = True
    print("Google Maps library loaded.")
except ImportError:
    GMAPS_AVAILABLE = False
    print("Google Maps not available.")

OR-Tools loaded successfully.
Google Maps library loaded.


## Section 3 — Model Constants

All values are sourced directly from the RH case document.  
Changing any value here automatically recalculates the entire model.

---

### Truck & Operations
| Parameter | Value | Source |
|-----------|-------|--------|
| Truck capacity | 1,700 cu ft / 7,000 lbs | RH case — 26-foot cargo box |
| Delivery window | Tuesday – Saturday, 8:00 AM – 5:00 PM | RH case — Costs and Logistics |

### Crew & Labor
| Parameter | Value | Source |
|-----------|-------|--------|
| Crew size | 3 persons (1 driver + 2 crew) | RH case — Costs and Logistics |
| Base wage | $25.00 / hr per crew member | RH case — Costs and Logistics |
| Driver premium | +20% → $30.00 / hr | RH case — Costs and Logistics |
| Benefits multiplier | 1.30× on total wages | RH case — health + payroll taxes |
| Pay rounding | 15-min intervals, round up at 7-min mark | RH case — Costs and Logistics |

### Cost Rates
| Parameter | Value | Source |
|-----------|-------|--------|
| Truck operating cost | $0.74 – $1.67 / mile → **$1.205 midpoint used** | Exhibit III — Henry 2020 |
| Holding cost — small items | $5.00 / item / day | RH case — Costs and Logistics |
| Holding cost — large items | $10.00 / item / day | RH case — Costs and Logistics |

### CO₂ Emissions (U.S. EPA)
| Parameter | Value | Source |
|-----------|-------|--------|
| Driving emissions | 1.69 kg CO₂ / mile | EPA — medium-duty freight vehicles |
| Idling emissions | 3.97 kg CO₂ / hr | EPA — diesel idle emission factor |

In [ ]:
# ── Warehouse ──────────────────────────────────────────────────────────────────
DC_ID      = "DC"
DC_ADDRESS = "590 20th St, San Francisco, CA 94107"

# ── Truck constraints (RH case — 26-foot cargo box) ───────────────────────────
TRUCK_CAPACITY_CUFT = 1700   # cubic feet
TRUCK_CAPACITY_LBS  = 7000   # pounds

# ── Operating hours (minutes from 8:00 AM) ────────────────────────────────────
SHIFT_END_MIN = 540          # 9 hours → 5:00 PM

# ── Fixed time costs — Exhibit V, RH case ─────────────────────────────────────
CHECKIN_MIN          = 23    # start-of-day briefing, once per day
COORD_PER_STOP_MIN   = 6     # arrival coordination per customer stop
SIGNOFF_PER_STOP_MIN = 8     # sign-off / photos per customer stop

# ── Crew wages — RH case, Costs and Logistics ─────────────────────────────────
CREW_BASE_WAGE      = 25.00  # $/hr per crew member
DRIVER_WAGE         = CREW_BASE_WAGE * 1.20   # $30/hr (20% premium)
BENEFITS_MULTIPLIER = 1.30   # health coverage + payroll taxes

# ── Pay rounding — RH case ────────────────────────────────────────────────────
PAY_INTERVAL_HRS  = 0.25     # 15-minute intervals
PAY_THRESHOLD_MIN = 7        # round up if ≥ 7 min past last quarter

# ── Truck operating cost — Exhibit III, Henry 2020 ────────────────────────────
TRUCK_COST_LOW  = 0.74       # $/mile low estimate
TRUCK_COST_HIGH = 1.67       # $/mile high estimate
TRUCK_COST_MID  = (TRUCK_COST_LOW + TRUCK_COST_HIGH) / 2   # $1.205/mile

# ── Inventory holding costs — RH case ─────────────────────────────────────────
HOLDING_SMALL = 5.00         # $/item/day — chairs, lamps, nightstands, stools
HOLDING_LARGE = 10.00        # $/item/day — beds, sofas, tables, cabinets

# ── CO₂ emission factors — U.S. EPA ──────────────────────────────────────────
CO2_PER_MILE = 1.69          # kg CO₂ per mile driven
CO2_PER_HOUR = 3.97          # kg CO₂ per hour idling

# ── Average urban speed ───────────────────────────────────────────────────────
AVG_SPEED_MPH = 25

print("Constants loaded.")
print(f"  Truck capacity:  {TRUCK_CAPACITY_CUFT:,} cu ft | {TRUCK_CAPACITY_LBS:,} lbs")
print(f"  Shift window:    8:00 AM – 5:00 PM ({SHIFT_END_MIN} min)")
print(f"  Labor rate:      2 crew × ${CREW_BASE_WAGE}/hr + 1 driver × ${DRIVER_WAGE}/hr × {BENEFITS_MULTIPLIER} benefits")
print(f"  Truck cost:      ${TRUCK_COST_MID:.3f}/mile (midpoint of ${TRUCK_COST_LOW}–${TRUCK_COST_HIGH} range)")

Constants loaded.
  Truck capacity:  1,700 cu ft | 7,000 lbs
  Shift window:    8:00 AM – 5:00 PM (540 min)
  Labor rate:      2 crew × $25.0/hr + 1 driver × $30.0/hr × 1.3 benefits
  Truck cost:      $1.205/mile (midpoint of $0.74–$1.67 range)


## Section 4 — Loading Source Files

Upload the three files when prompted:  
1. `RH Customer Orders  Dataset_10.csv` — the order data  
2. `Furniture Details - Unlimited Furniture Deilvery at RH.xlsx` — dimensions reference (rh.com)  
3. `Unlimited Furniture Delivery at RH - Customers Addresses.xlsx` — customer address database

In [ ]:
# ── Load orders ────────────────────────────────────────────────────────────────
df_orders = pd.read_csv("RH Customer Orders  Dataset_10.csv")
df_orders.columns = ['Customer_ID', 'Brand', 'Product_Description', 'Quantity']
print(f"Orders loaded: {df_orders.shape[0]} rows, {df_orders['Customer_ID'].nunique()} customers")

Orders loaded: 168 rows, 19 customers


In [ ]:
# ── Load furniture dimensions file (for reference) ─────────────────────────────
# Note: Dimensions are pre-loaded into the DIMS dictionary in Section 5
# to ensure exact key matching with the order CSV. The furniture file uses
# different product descriptions than the order CSV causing zero matches
# when parsed dynamically. DIMS keys match the order CSV exactly.
df_furniture_raw = pd.read_excel("Furniture Details - Unlimited Furniture Deilvery at RH.xlsx", header=None)
print(f"Furniture file loaded: {df_furniture_raw.shape[0]} rows (used as reference source for DIMS dictionary)")

Furniture file loaded: 55 rows (used as reference source for DIMS dictionary)


In [ ]:
# ── Load customer addresses ────────────────────────────────────────────────────
df_addresses = pd.read_excel("Unlimited Furniture Delivery at RH - Customers Addresses.xlsx", usecols=[0, 1])
df_addresses.columns = ['Customer_ID', 'Address']
df_addresses = df_addresses.dropna(subset=['Customer_ID', 'Address'])
print(f"Address database loaded: {len(df_addresses)} customers")

Address database loaded: 372 customers


### 4a — Order data preview

In [ ]:
print(f"Shape: {df_orders.shape}")
df_orders.head(10)

Shape: (168, 4)


,Customer_ID,Brand,Product_Description,Quantity
0,CUST5881,Reclaimed Oak Reeded Round Dining Table,48'' Table,1
1,CUST5881,Nicola Track Vegan Leather Dining Side Chair,Dining Chair,4
2,CUST5881,Mycena Obelisk Stone Table Lamp,24'' Lamp,1
3,CUST5881,Orizaba Rectangular Fire Table,72'' W Fire Table,3
4,CUST5881,Maxwell Leather Right-Arm Bench-Seat L-Sectional,Luxe 12'' Sectional,3
5,CUST5881,Terreno Wall Fountain 22'',Carbon Fountain,2
6,CUST5881,Maxwell Leather Right-Arm Bench-Seat L-Sectional,Classic 96'' Sectional,1
7,CUST7841,Reclaimed Oak Reeded Round Dining Table,72'' Table,1
8,CUST7841,Nicola Track Vegan Leather Dining Side Chair,Dining Chair,8
9,CUST7841,Reclaimed Oak Reeded Round Dining Table,60'' Table,1


### 4b — Address database preview

In [ ]:
print(f"Shape: {df_addresses.shape}")
df_addresses.head(10)

Shape: (372, 2)


,Customer_ID,Address
0,CUST1011,"457 Diller Street, Alameda, CA 94501"
1,CUST1064,"2164 San Jose Avenue, Alameda, CA 94501"
2,CUST1070,"1535 Saint Charles Street, Alameda, CA 94501"
3,CUST1075,"85 Vicente Rd, Berkeley, CA 94705"
4,CUST1133,13397 Campus Drive – 94619 Oakland
5,CUST1155,"945 Green St UNIT 8, San Francisco, CA 94133"
6,CUST1162,"1325 Glendale Ave, Berkeley, CA 94708"
7,CUST1174,"105 3rd Street #2, Sausalito, CA 94965"
8,CUST1201,"101 Sacramento Way, Sausalito, CA 94965"
9,CUST1202,"765 Eagle Avenue, Alameda, CA 94501"


## Section 5 — Product Dimensions (DIMS Dictionary)

All 168 order lines are mapped using a direct lookup dictionary.  
Keys are `(Brand, Product_Description)` exactly as they appear in the order CSV.  
Values are `(vol_cuft, weight_lbs, load_min, unload_min, size_class)`.

**Sources:**
- Volumes and weights: rh.com product pages (via Furniture Details file)
- Load/unload times: Exhibit V of the RH case document

| Item Type | Load (min/item) | Unload & Stage (min/item) |
|-----------|----------------|---------------------------|
| Chair / Stool | 1.0 | 2.0 |
| Nightstand | 1.5 | 2.5 |
| Lamp | 1.5 | 2.0 |
| Sofa / Sectional | 2.5 | 4.0 |
| Dining & Fire Table | 3.0 | 5.0 |
| Bed Frame | 6.0 | 8.0 |
| Cabinet / Fountain | 4.0 | 12.0 |

In [ ]:
# Format: (Brand, Product_Description) -> (vol_cuft, weight_lbs, load_min, unload_min, size_class)
# Volumes/weights: rh.com  |  Times: Exhibit V, RH case document

DIMS = {
    # Aero Wood Round Dining Table
    ("Aero Wood Round Dining Table", "30'' Bistro Table"):  (12.27,  31, 3.0, 5.0, "large"),
    ("Aero Wood Round Dining Table", "36'' Bistro Table"):  (17.67,  51, 3.0, 5.0, "large"),
    ("Aero Wood Round Dining Table", "42'' Bistro Table"):  (24.05,  57, 3.0, 5.0, "large"),
    ("Aero Wood Round Dining Table", "50'' Bistro Table"):  (33.52,  72, 3.0, 5.0, "large"),
    # Arno Fabric Bar & Counter Stool
    ("Arno Fabric Bar & Counter Stool", "Counter Stool 36 1/4'' H"): (8.27, 19, 1.0, 2.0, "small"),
    ("Arno Fabric Bar & Counter Stool", "Barstool 40 1/4'' H"):      (9.18, 20, 1.0, 2.0, "small"),
    # Arno Fabric Dining Armchair
    ("Arno Fabric Dining Armchair", "Dining Armchair"): (8.45, 17, 1.0, 2.0, "small"),
    ("Arno Fabric Dining Armchair", "Dining Chair"):    (8.45, 17, 1.0, 2.0, "small"),
    # Arno Vegan Leather Dining Side Chair
    ("Arno Vegan Leather Dining Side Chair", "Dining Side Chair"): (7.87, 15, 1.0, 2.0, "small"),
    # Aurelie Fabric Dining Side Chair
    ("Aurelie Fabric Dining Side Chair", "Dining Side Chair"): (7.08, 36, 1.0, 2.0, "small"),
    # Byron Canopy Bed
    ("Byron Canopy Bed", "King Bed 90''"):      (367.29, 480, 6.0, 8.0, "large"),
    ("Byron Canopy Bed", "Cal King Bed 90''"):  (365.62, 475, 6.0, 8.0, "large"),
    ("Byron Canopy Bed", "King Bed 102''"):     (416.26, 510, 6.0, 8.0, "large"),
    ("Byron Canopy Bed", "Cal King Bed 102''"): (414.38, 507, 6.0, 8.0, "large"),
    # Byron Emperador Dining Table
    ("Byron Emperador Dining Table", "72'' Dining Table"): (50.00, 462, 3.0, 5.0, "large"),
    ("Byron Emperador Dining Table", "84'' Dining Table"): (58.33, 512, 3.0, 5.0, "large"),
    ("Byron Emperador Dining Table", "96'' Dining Table"): (66.67, 484, 3.0, 5.0, "large"),
    # Kyoto Closed Nightstand
    ("Kyoto Closed Nightstand", "18'' Nightstand"): (4.12, 33, 1.5, 2.5, "small"),
    ("Kyoto Closed Nightstand", "22'' Nightstand"): (5.04, 41, 1.5, 2.5, "small"),
    ("Kyoto Closed Nightstand", "26'' Nightstand"): (6.50, 52, 1.5, 2.5, "small"),
    ("Kyoto Closed Nightstand", "32'' Nightstand"): (8.00, 64, 1.5, 2.5, "small"),
    # Ligné Panel Bed with Footboard
    ("Ligné Panel Bed with Footboard", "Queen Bed"):    (180.81, 398, 6.0, 8.0, "large"),
    ("Ligné Panel Bed with Footboard", "King Bed"):     (225.66, 458, 6.0, 8.0, "large"),
    ("Ligné Panel Bed with Footboard", "Cal King Bed"): (224.36, 449, 6.0, 8.0, "large"),
    # Maxwell Leather Right-Arm Bench-Seat L-Sectional
    ("Maxwell Leather Right-Arm Bench-Seat L-Sectional", "Classic 72'' Sectional"):  (56.67,  397, 2.5, 4.0, "large"),
    ("Maxwell Leather Right-Arm Bench-Seat L-Sectional", "Classic 96'' Sectional"):  (75.56,  529, 2.5, 4.0, "large"),
    ("Maxwell Leather Right-Arm Bench-Seat L-Sectional", "Luxe 108'' Sectional"):    (97.75,  684, 2.5, 4.0, "large"),
    ("Maxwell Leather Right-Arm Bench-Seat L-Sectional", "Luxe 12'' Sectional"):    (108.67,  761, 2.5, 4.0, "large"),
    # Maxwell Leather Sofa
    ("Maxwell Leather Sofa", "Classic 6' Sofa"): (56.67, 453, 2.5, 4.0, "large"),
    ("Maxwell Leather Sofa", "Classic 8' Sofa"): (75.56, 604, 2.5, 4.0, "large"),
    ("Maxwell Leather Sofa", "Luxe 7' Sofa"):    (86.89, 608, 2.5, 4.0, "large"),
    ("Maxwell Leather Sofa", "Luxe 9' Sofa"):    (97.75, 648, 2.5, 4.0, "large"),
    # Melbourne Bar Cabinet
    ("Melbourne Bar Cabinet", "Bar Cabinet"): (17.50, 211, 4.0, 12.0, "large"),
    # Mycena Obelisk Stone Table Lamp
    ("Mycena Obelisk Stone Table Lamp", "19'' Lamp"): (2.31, 15, 1.5, 2.0, "small"),
    ("Mycena Obelisk Stone Table Lamp", "24'' Lamp"): (4.50, 29, 1.5, 2.0, "small"),
    # Nicola Track Vegan Leather Dining Side Chair
    ("Nicola Track Vegan Leather Dining Side Chair", "Dining Chair"): (9.25, 16, 1.0, 2.0, "small"),
    # Noemi Torchiere Floor Lamp
    ("Noemi Torchiere Floor Lamp", "Burnished Brass"): (7.18, 21, 1.5, 2.0, "small"),
    # Orizaba Rectangular Fire Table
    ("Orizaba Rectangular Fire Table", "56'' W Fire Table"): (16.33, 325, 3.0, 5.0, "large"),
    ("Orizaba Rectangular Fire Table", "72'' W Fire Table"): (21.00, 345, 3.0, 5.0, "large"),
    # Reclaimed Oak Reeded Round Dining Table
    ("Reclaimed Oak Reeded Round Dining Table", "48'' Table"): (31.42, 306, 3.0, 5.0, "large"),
    ("Reclaimed Oak Reeded Round Dining Table", "60'' Table"): (49.09, 384, 3.0, 5.0, "large"),
    ("Reclaimed Oak Reeded Round Dining Table", "72'' Table"): (70.69, 438, 3.0, 5.0, "large"),
    # Terreno Wall Fountain 22''
    ("Terreno Wall Fountain 22''", "Carbon Fountain"): (10.72, 23, 4.0, 12.0, "large"),
    # Thaddeus Curved Leather Dining Armchair
    ("Thaddeus Curved Leather Dining Armchair", "Dining Armchair"): (7.18, 40, 1.0, 2.0, "small"),
}

print(f"DIMS lookup: {len(DIMS)} SKUs defined.")
print("All volumes and weights from rh.com. All times from Exhibit V, RH case document.")

DIMS lookup: 44 SKUs defined.
All volumes and weights from rh.com. All times from Exhibit V, RH case document.


## Section 6 — Matching Orders to Customers and Addresses

In [ ]:
# ── Step 1: Identify the 19 customers in this dataset ─────────────────────────
order_customers = df_orders['Customer_ID'].unique()
print(f"Customers in order dataset: {len(order_customers)}")
print(sorted(order_customers))

Customers in order dataset: 19
['CUST1202', 'CUST2394', 'CUST3601', 'CUST4656', 'CUST4775', 'CUST5141', 'CUST5146', 'CUST5881', 'CUST6386', 'CUST6463', 'CUST6840', 'CUST7287', 'CUST7291', 'CUST7594', 'CUST7841', 'CUST7884', 'CUST8298', 'CUST8401', 'CUST9328']


In [ ]:
# ── Step 2: Look up addresses for these 19 customers ──────────────────────────
addr_lookup = df_addresses.set_index('Customer_ID')['Address'].to_dict()

customer_addresses = {}
missing_addresses  = []
for cid in order_customers:
    if cid in addr_lookup:
        customer_addresses[cid] = addr_lookup[cid]
    else:
        missing_addresses.append(cid)

print(f"Addresses found:   {len(customer_addresses)}")
if missing_addresses:
    print(f"Addresses missing: {missing_addresses}")
else:
    print("All 19 customers matched to addresses successfully.")

Addresses found:   19
All 19 customers matched to addresses successfully.


In [ ]:
# Display matched addresses
pd.DataFrame([
    {'Customer_ID': k, 'Address': v}
    for k, v in sorted(customer_addresses.items())
])

,Customer_ID,Address
0,CUST1202,"765 Eagle Avenue, Alameda, CA 94501"
1,CUST2394,"1615 La Vereda Rd, Berkeley, CA 94709"
2,CUST3601,"1416 Broadway, Alameda, CA 94501"
3,CUST4656,"555 Pierce St APT 408, Albany, CA 94706"
4,CUST4775,13447 Campus Drive – 94619 Oakland
5,CUST5141,"9101 Washington St, San Francisco, CA 94118 (..."
6,CUST5146,"233 Village Way, South San Francisco, CA 94080"
7,CUST5881,13407 Campus Drive – 94619 Oakland
8,CUST6386,"1820 3rd Street, Alameda, CA 94501"
9,CUST6463,"2712 Calhoun Street, Alameda, CA 94501"


## Section 7 — Computing Per-Customer Order Totals

For each customer we calculate:
- **Total volume** (cu ft) — sum of (unit volume × quantity)
- **Total weight** (lbs) — sum of (unit weight × quantity)
- **Number of items** — total quantity across all order lines
- **Load time** (min) — time to load at warehouse (Exhibit V)
- **Service time** (min) — 6 min coordination + unload/stage time + 8 min sign-off

In [ ]:
# ── Merge orders with DIMS lookup ─────────────────────────────────────────────
records = []
unmatched_rows = []

for _, row in df_orders.iterrows():
    key = (row['Brand'], row['Product_Description'])
    if key in DIMS:
        vol, wt, load_t, unload_t, size_class = DIMS[key]
        records.append({
            'Customer_ID':         row['Customer_ID'],
            'Brand':               row['Brand'],
            'Product_Description': row['Product_Description'],
            'Quantity':            row['Quantity'],
            'Vol_CuFt':            vol,
            'Weight_Lbs':          wt,
            'Load_Min':            load_t,
            'Unload_Min':          unload_t,
            'Size_Class':          size_class,
        })
    else:
        unmatched_rows.append(row)

df_merged = pd.DataFrame(records)

if unmatched_rows:
    print(f"WARNING: {len(unmatched_rows)} unmatched — check DIMS dictionary")
    for r in unmatched_rows:
        print(f"  {r['Customer_ID']} | {r['Brand']} | {r['Product_Description']}")
else:
    print(f"All {len(df_merged)} order lines matched — zero warnings.")

All 168 order lines matched — zero warnings.


In [ ]:
# ── Compute totals per customer ────────────────────────────────────────────────
df_merged['Total_Vol']    = df_merged['Vol_CuFt']    * df_merged['Quantity']
df_merged['Total_Wt']     = df_merged['Weight_Lbs']  * df_merged['Quantity']
df_merged['Total_Load']   = df_merged['Load_Min']    * df_merged['Quantity']
df_merged['Total_Unload'] = df_merged['Unload_Min']  * df_merged['Quantity']
df_merged['Small_Items']  = df_merged['Quantity'].where(df_merged['Size_Class'] == 'small', 0)
df_merged['Large_Items']  = df_merged['Quantity'].where(df_merged['Size_Class'] == 'large', 0)

df_customers = df_merged.groupby('Customer_ID').agg(
    Total_Vol_CuFt  = ('Total_Vol',    'sum'),
    Total_Wt_Lbs    = ('Total_Wt',     'sum'),
    Num_Items       = ('Quantity',     'sum'),
    Load_Time_Min   = ('Total_Load',   'sum'),
    Unload_Time_Min = ('Total_Unload', 'sum'),
    Small_Items     = ('Small_Items',  'sum'),
    Large_Items     = ('Large_Items',  'sum'),
).reset_index()

# Service time = 6 min arrival coordination + unload time + 8 min sign-off (Exhibit V)
df_customers['Service_Time_Min'] = (
    COORD_PER_STOP_MIN + df_customers['Unload_Time_Min'] + SIGNOFF_PER_STOP_MIN
)
df_customers['Address']        = df_customers['Customer_ID'].map(customer_addresses)
df_customers['Exceeds_Volume'] = df_customers['Total_Vol_CuFt'] > TRUCK_CAPACITY_CUFT
df_customers['Exceeds_Weight'] = df_customers['Total_Wt_Lbs']  > TRUCK_CAPACITY_LBS

print(f"Per-customer summary ({len(df_customers)} customers):")
df_customers[['Customer_ID','Total_Vol_CuFt','Total_Wt_Lbs','Num_Items',
              'Load_Time_Min','Service_Time_Min','Exceeds_Volume','Exceeds_Weight']]

Per-customer summary (19 customers):


,Customer_ID,Total_Vol_CuFt,Total_Wt_Lbs,Num_Items,Load_Time_Min,Service_Time_Min,Exceeds_Volume,Exceeds_Weight
0,CUST1202,1671.75,4895,32,63.0,115.5,False,False
1,CUST2394,2155.87,5007,32,75.5,139.0,True,False
2,CUST3601,419.00,537,3,9.0,26.0,False,False
3,CUST4656,221.45,1074,18,25.0,65.0,False,False
4,CUST4775,1768.57,6421,35,61.5,119.0,True,False
5,CUST5141,144.69,566,9,11.0,35.0,False,False
6,CUST5146,785.33,2695,23,42.0,90.5,False,False
7,CUST5881,558.93,4292,15,35.5,84.0,False,False
8,CUST6386,16.33,325,1,3.0,19.0,False,False
9,CUST6463,215.21,702,14,18.0,48.0,False,False


In [ ]:
# ── Flag customers requiring solo delivery days ────────────────────────────────
solo_customers = df_customers[
    df_customers['Exceeds_Volume'] | df_customers['Exceeds_Weight']
]['Customer_ID'].tolist()

print("Customers requiring dedicated solo delivery days:")
for cid in solo_customers:
    row = df_customers[df_customers['Customer_ID'] == cid].iloc[0]
    reasons = []
    if row['Exceeds_Volume']: reasons.append(f"volume {row['Total_Vol_CuFt']:.0f} cu ft > {TRUCK_CAPACITY_CUFT}")
    if row['Exceeds_Weight']: reasons.append(f"weight {row['Total_Wt_Lbs']:.0f} lbs > {TRUCK_CAPACITY_LBS}")
    print(f"  {cid}: {', '.join(reasons)}")

Customers requiring dedicated solo delivery days:
  CUST2394: volume 2156 cu ft > 1700
  CUST4775: volume 1769 cu ft > 1700
  CUST9328: volume 1892 cu ft > 1700


## Section 8 — Distance Matrix

Real driving distances pulled once from the **Google Maps Distance Matrix API** and hardcoded below.  
To refresh: set `GOOGLE_MAPS_API_KEY` and call `build_distance_matrix_gmaps()`.

In [ ]:
# ── Google Maps API key (set to refresh distances) ────────────────────────────
GOOGLE_MAPS_API_KEY = "xxxxxxxxxxxxxxxxxxxxxxxxxxx"  # replace to refresh

# ── Node order: DC first, then customers sorted ───────────────────────────────
ALL_NODES = [DC_ID] + sorted(customer_addresses.keys())
NODE_INDEX = {node: i for i, node in enumerate(ALL_NODES)}
N = len(ALL_NODES)

# ── Hardcoded distance matrix (miles) — Google Maps Distance Matrix API Apr 2026
# Node order: DC, CUST1202, CUST2394, CUST3601, CUST4656, CUST4775,
#             CUST5141, CUST5146, CUST5881, CUST6386, CUST6463,
#             CUST6840, CUST7287, CUST7291, CUST7594, CUST7841,
#             CUST7884, CUST8298, CUST8401, CUST9328
DIST_MATRIX = [
    [ 0.0, 13.6, 14.9, 15.9, 13.9, 19.8,  5.7,  8.7, 19.9, 14.1, 16.6, 13.8, 15.3,  2.9, 16.2,  2.9,  9.3, 14.4,  3.0, 11.1],
    [14.0,  0.0,  9.1,  2.6, 10.7, 13.5, 15.8, 21.8, 13.6,  1.2,  2.8,  0.5,  2.3, 11.9,  1.9, 11.9, 22.4,  8.2, 13.0,  7.9],
    [15.2,  8.7,  0.0, 11.2,  4.2,  9.8, 16.9, 22.9,  9.8,  9.3, 11.8,  9.0, 10.5, 13.1, 11.4, 13.1, 23.5,  4.3, 14.2,  4.3],
    [16.2,  2.6, 11.2,  0.0, 12.8,  6.3, 17.9, 24.0,  6.2,  3.2,  0.7,  2.7,  0.3, 14.1,  1.0, 14.1, 24.6, 10.4, 15.2, 10.1],
    [14.2, 10.4,  4.2, 13.0,  0.0, 16.1, 15.9, 21.9, 16.1, 11.0, 13.6, 10.7, 12.3, 12.1, 13.2, 12.1, 22.5, 10.7, 13.2,  5.0],
    [20.0, 13.3,  9.8,  6.3, 15.8,  0.0, 21.7, 27.7,  0.1, 13.9,  7.0, 13.6,  6.4, 17.9,  7.2, 17.9, 28.3,  6.3, 19.0, 13.0],
    [ 6.6, 17.0, 18.3, 19.3, 17.3, 23.2,  0.0, 13.2, 23.3, 17.6, 20.0, 17.2, 18.7,  4.2, 19.6,  4.2, 13.8, 17.8,  2.9, 14.5],
    [ 9.3, 21.6, 22.9, 23.9, 21.9, 27.9, 13.3,  0.0, 27.9, 22.2, 24.6, 21.9, 23.3, 11.0, 24.2, 11.0,  0.8, 22.5, 11.6, 19.1],
    [20.0, 13.4,  9.7,  6.3, 15.9,  0.1, 21.8, 27.8,  0.0, 13.9,  6.9, 13.6,  6.3, 17.9,  7.1, 17.9, 28.4,  6.2, 19.0, 13.1],
    [14.7,  1.0,  9.7,  3.2, 11.3, 14.1, 16.4, 22.4, 14.2,  0.0,  3.6,  0.5,  2.9, 12.5,  2.5, 12.5, 23.0,  8.8, 13.6,  8.5],
    [16.8,  2.8, 11.8,  0.7, 13.4,  6.7, 18.5, 24.5,  6.6,  3.5,  0.0,  3.0,  1.0, 14.7,  1.1, 14.7, 25.1, 11.0, 15.8, 10.6],
    [14.3,  0.5,  9.3,  2.8, 10.9, 13.8, 16.0, 22.1, 13.9,  0.5,  3.0,  0.0,  2.5, 12.2,  2.0, 12.2, 22.7,  8.5, 13.3,  8.1],
    [15.6,  2.2, 10.6,  0.3, 12.2,  6.6, 17.3, 23.4,  6.5,  2.9,  1.0,  2.5,  0.0, 13.5,  0.8, 13.5, 23.9,  9.8, 14.6,  9.4],
    [ 3.3, 11.9, 13.2, 14.2, 12.2, 18.1,  3.8, 11.1, 18.2, 12.4, 14.8, 12.1, 13.6,  0.0, 14.4,  0.0, 11.6, 12.7,  1.1,  9.4],
    [16.5,  1.8, 11.5,  1.0, 13.1,  7.5, 18.2, 24.2,  7.4,  2.5,  1.1,  2.0,  0.8, 14.4,  0.0, 14.4, 24.8, 10.7, 15.5, 10.3],
    [ 3.3, 11.9, 13.2, 14.2, 12.2, 18.1,  3.8, 11.1, 18.2, 12.4, 14.8, 12.1, 13.6,  0.4, 14.4,  0.0, 11.6, 12.7,  1.1,  9.4],
    [10.0, 22.4, 23.7, 24.7, 22.6, 28.6, 13.3,  1.0, 28.6, 22.9, 25.3, 22.6, 24.1, 11.7, 24.9, 11.7,  0.0, 23.2, 12.3, 19.9],
    [14.9,  8.3,  4.5, 10.7, 10.7,  6.4, 16.6, 22.6,  6.3,  8.9, 11.4,  8.6, 10.1, 12.7, 11.0, 12.7, 23.2,  0.0, 13.8,  5.4],
    [ 3.2, 12.6, 13.9, 14.9, 12.9, 18.8,  3.0, 11.4, 18.9, 13.2, 15.6, 12.8, 14.3,  1.3, 15.2,  1.3, 11.9, 13.4,  0.0, 10.1],
    [10.9,  7.1,  4.3,  9.7,  4.4, 12.8, 12.6, 18.6, 12.9,  7.7, 10.3,  7.4,  9.1,  8.8,  9.9,  8.8, 19.2,  5.0,  9.9,  0.0],
]

# ── Travel time matrix (minutes) — Google Maps Distance Matrix API Apr 2026 ───
TIME_MATRIX = [
    [ 0,  26,  30,  24,  22,  27,  26,  15,  27,  28,  27,  27,  24,  14,  27,  14,  17,  21,  17,  20],
    [ 27,  0,  22,   7,  25,  18,  33,  40,  18,   4,   8,   3,   6,  24,   5,  24,  41,  17,  26,  16],
    [ 30, 22,   0,  26,  12,  17,  37,  44,  17,  23,  27,  22,  25,  28,  26,  28,  44,  10,  31,  11],
    [ 25,  7,  26,   0,  29,  12,  32,  39,  12,   8,   2,   7,   1,  18,   2,  18,  39,  21,  19,  20],
    [ 23, 25,  12,  29,   0,  23,  35,  42,  23,  26,  30,  25,  28,  22,  29,  22,  42,  22,  24,  14],
    [ 27, 18,  17,  12,  23,   0,  39,  46,   1,  19,  13,  18,  11,  25,  12,  25,  47,  11,  26,  18],
    [ 28, 35,  39,  34,  37,  42,   0,  23,  42,  36,  40,  35,  38,  16,  39,  16,  24,  38,  14,  30],
    [ 16, 40,  44,  39,  42,  47,  24,   0,  47,  41,  45,  40,  43,  21,  44,  21,   2,  42,  19,  35],
    [ 27, 18,  17,  12,  24,   0,  40,  46,   0,  19,  13,  18,  11,  25,  12,  25,  47,  11,  26,  18],
    [ 28,  4,  23,   8,  26,  19,  34,  41,  19,   0,   9,   3,   7,  25,   6,  25,  42,  18,  27,  17],
    [ 27,  8,  27,   2,  30,  13,  35,  45,  13,   9,   0,   8,   2,  22,   2,  22,  45,  22,  20,  21],
    [ 27,  3,  22,   7,  25,  18,  33,  40,  18,   3,   8,   0,   6,  24,   5,  24,  41,  17,  25,  16],
    [ 25,  6,  25,   1,  28,  11,  33,  43,  11,   7,   2,   6,   0,  20,   1,  20,  43,  20,  18,  19],
    [ 14, 24,  28,  18,  22,  25,  16,  22,  25,  25,  22,  24,  20,   0,  20,   1,  22,  25,   8,  20],
    [ 27,  5,  26,   2,  29,  12,  34,  44,  12,   6,   2,   5,   1,  20,   0,  20,  44,  21,  18,  20],
    [ 14, 24,  28,  18,  22,  25,  16,  22,  25,  25,  22,  24,  20,   1,  20,   0,  22,  25,   8,  20],
    [ 18, 41,  44,  39,  42,  48,  24,   2,  48,  42,  45,  41,  43,  22,  44,  22,   0,  42,  20,  36],
    [ 21, 17,  10,  21,  22,  11,  28,  42,  11,  18,  22,  17,  20,  22,  21,  22,  43,   0,  22,  10],
    [ 17, 26,  31,  19,  24,  26,  14,  19,  26,  27,  20,  25,  18,   8,  19,   8,  20,  22,   0,  18],
    [ 20, 16,  11,  20,  14,  18,  30,  35,  18,  17,  21,  16,  19,  20,  20,  20,  36,  10,  18,   0],
]

print(f"Distance matrix: {N}×{N} nodes loaded")
print("Source: Google Maps Distance Matrix API (pulled April 2026)\n")
print("DC → each customer:")
for j, node in enumerate(ALL_NODES[1:], 1):
    print(f"  DC → {node}: {DIST_MATRIX[0][j]:.1f} miles | {TIME_MATRIX[0][j]:.0f} min")

Distance matrix: 20×20 nodes loaded
Source: Google Maps Distance Matrix API (pulled April 2026)

DC → each customer:
  DC → CUST1202: 13.6 miles | 26 min
  DC → CUST2394: 14.9 miles | 30 min
  DC → CUST3601: 15.9 miles | 24 min
  DC → CUST4656: 13.9 miles | 22 min
  DC → CUST4775: 19.8 miles | 27 min
  DC → CUST5141: 5.7 miles | 26 min
  DC → CUST5146: 8.7 miles | 15 min
  DC → CUST5881: 19.9 miles | 27 min
  DC → CUST6386: 14.1 miles | 28 min
  DC → CUST6463: 16.6 miles | 27 min
  DC → CUST6840: 13.8 miles | 27 min
  DC → CUST7287: 15.3 miles | 24 min
  DC → CUST7291: 2.9 miles | 14 min
  DC → CUST7594: 16.2 miles | 27 min
  DC → CUST7841: 2.9 miles | 14 min
  DC → CUST7884: 9.3 miles | 17 min
  DC → CUST8298: 14.4 miles | 21 min
  DC → CUST8401: 3.0 miles | 17 min
  DC → CUST9328: 11.1 miles | 20 min


## Section 8b — Calendar Assignment

Deliveries run Tuesday through Saturday. Day 1 = Tuesday April 7, 2026.

In [ ]:
from datetime import date, timedelta

START_DATE   = date(2026, 4, 7)   # First Tuesday
DAYS_OF_WEEK = ['Tuesday','Wednesday','Thursday','Friday','Saturday']

def day_num_to_calendar(day_num):
    """Map Day 1→Tue Apr 7, Day 2→Wed Apr 8, Day 6→Tue Apr 14, etc."""
    week, position = divmod(day_num - 1, 5)
    delivery_date  = START_DATE + timedelta(weeks=week, days=position)
    return delivery_date.strftime('%Y-%m-%d'), DAYS_OF_WEEK[position]

print("Planned delivery calendar:")
for d in range(1, 10):
    dt, dn = day_num_to_calendar(d)
    print(f"  Day {d}: {dn} {dt}")

Planned delivery calendar:
  Day 1: Tuesday 2026-04-07
  Day 2: Wednesday 2026-04-08
  Day 3: Thursday 2026-04-09
  Day 4: Friday 2026-04-10
  Day 5: Saturday 2026-04-11
  Day 6: Tuesday 2026-04-14
  Day 7: Wednesday 2026-04-15
  Day 8: Thursday 2026-04-16
  Day 9: Friday 2026-04-17


## Section 8c — Distance Matrix Heatmap

In [ ]:
df_dist = pd.DataFrame(DIST_MATRIX, index=ALL_NODES, columns=ALL_NODES)
print("Distance matrix (miles):")
df_dist.style.format("{:.1f}").background_gradient(cmap='YlOrRd', axis=None)

Distance matrix (miles):


,DC,CUST1202,CUST2394,CUST3601,CUST4656,CUST4775,CUST5141,CUST5146,CUST5881,CUST6386,CUST6463,CUST6840,CUST7287,CUST7291,CUST7594,CUST7841,CUST7884,CUST8298,CUST8401,CUST9328
DC,0.0,13.6,14.9,15.9,13.9,19.8,5.7,8.7,19.9,14.1,16.6,13.8,15.3,2.9,16.2,2.9,9.3,14.4,3.0,11.1
CUST1202,14.0,0.0,9.1,2.6,10.7,13.5,15.8,21.8,13.6,1.2,2.8,0.5,2.3,11.9,1.9,11.9,22.4,8.2,13.0,7.9
CUST2394,15.2,8.7,0.0,11.2,4.2,9.8,16.9,22.9,9.8,9.3,11.8,9.0,10.5,13.1,11.4,13.1,23.5,4.3,14.2,4.3
CUST3601,16.2,2.6,11.2,0.0,12.8,6.3,17.9,24.0,6.2,3.2,0.7,2.7,0.3,14.1,1.0,14.1,24.6,10.4,15.2,10.1
CUST4656,14.2,10.4,4.2,13.0,0.0,16.1,15.9,21.9,16.1,11.0,13.6,10.7,12.3,12.1,13.2,12.1,22.5,10.7,13.2,5.0
CUST4775,20.0,13.3,9.8,6.3,15.8,0.0,21.7,27.7,0.1,13.9,7.0,13.6,6.4,17.9,7.2,17.9,28.3,6.3,19.0,13.0
CUST5141,6.6,17.0,18.3,19.3,17.3,23.2,0.0,13.2,23.3,17.6,20.0,17.2,18.7,4.2,19.6,4.2,13.8,17.8,2.9,14.5
CUST5146,9.3,21.6,22.9,23.9,21.9,27.9,13.3,0.0,27.9,22.2,24.6,21.9,23.3,11.0,24.2,11.0,0.8,22.5,11.6,19.1
CUST5881,20.0,13.4,9.7,6.3,15.9,0.1,21.8,27.8,0.0,13.9,6.9,13.6,6.3,17.9,7.1,17.9,28.4,6.2,19.0,13.1
CUST6386,14.7,1.0,9.7,3.2,11.3,14.1,16.4,22.4,14.2,0.0,3.6,0.5,2.9,12.5,2.5,12.5,23.0,8.8,13.6,8.5


## Section 9 — Assigning Customers to Daily Loads

**Rules:**
- Customers whose order exceeds 1,700 cu ft or 7,000 lbs get a dedicated solo day
- Remaining customers are packed greedily — largest orders first
- Each day must finish within 540 minutes (8 AM – 5 PM)
- The 23-minute morning check-in is deducted from each day's available time

In [ ]:
def assign_customers_to_days(df_cust):
    """Group customers into feasible daily loads. Returns list of lists."""
    cust_dict = df_cust.set_index('Customer_ID').to_dict('index')

    solo  = [[c] for c in df_cust[df_cust['Exceeds_Volume'] | df_cust['Exceeds_Weight']]['Customer_ID']]
    group = df_cust[
        ~(df_cust['Exceeds_Volume'] | df_cust['Exceeds_Weight'])
    ].sort_values('Total_Vol_CuFt', ascending=False)['Customer_ID'].tolist()

    day_loads = []
    while group:
        day, day_vol, day_wt = [], 0.0, 0.0
        day_time = CHECKIN_MIN
        remaining = []

        for cid in group:
            c = cust_dict[cid]
            ni = NODE_INDEX[cid]
            transit  = TIME_MATRIX[0][ni] + TIME_MATRIX[ni][0]
            time_add = c['Load_Time_Min'] + transit + c['Service_Time_Min']

            if (day_vol + c['Total_Vol_CuFt'] <= TRUCK_CAPACITY_CUFT and
                day_wt  + c['Total_Wt_Lbs']   <= TRUCK_CAPACITY_LBS  and
                day_time + time_add            <= SHIFT_END_MIN):
                day.append(cid)
                day_vol  += c['Total_Vol_CuFt']
                day_wt   += c['Total_Wt_Lbs']
                day_time += time_add
            else:
                remaining.append(cid)

        if not day and remaining:
            day.append(remaining.pop(0))
        if day:
            day_loads.append(day)
        group = remaining

    return solo + day_loads


all_days  = assign_customers_to_days(df_customers)
cust_dict = df_customers.set_index('Customer_ID').to_dict('index')

print(f"Delivery plan: {len(all_days)} days\n")
for i, day in enumerate(all_days, 1):
    vol     = sum(cust_dict[c]['Total_Vol_CuFt'] for c in day)
    wt      = sum(cust_dict[c]['Total_Wt_Lbs']  for c in day)
    pct_vol = vol / TRUCK_CAPACITY_CUFT * 100
    pct_wt  = wt  / TRUCK_CAPACITY_LBS  * 100
    print(f"  Day {i}: {day}")
    print(f"         Volume: {vol:.0f} cu ft ({pct_vol:.0f}%) | Weight: {wt:.0f} lbs ({pct_wt:.0f}%)")

Delivery plan: 9 days

  Day 1: ['CUST2394']
         Volume: 2156 cu ft (127%) | Weight: 5007 lbs (72%)
  Day 2: ['CUST4775']
         Volume: 1769 cu ft (104%) | Weight: 6421 lbs (92%)
  Day 3: ['CUST9328']
         Volume: 1892 cu ft (111%) | Weight: 6761 lbs (97%)
  Day 4: ['CUST1202', 'CUST6386']
         Volume: 1688 cu ft (99%) | Weight: 5220 lbs (75%)
  Day 5: ['CUST8401', 'CUST7841']
         Volume: 1655 cu ft (97%) | Weight: 5540 lbs (79%)
  Day 6: ['CUST5146', 'CUST7287']
         Volume: 1564 cu ft (92%) | Weight: 5040 lbs (72%)
  Day 7: ['CUST7594', 'CUST5881', 'CUST3601']
         Volume: 1600 cu ft (94%) | Weight: 6714 lbs (96%)
  Day 8: ['CUST6840', 'CUST8298', 'CUST7291', 'CUST6463']
         Volume: 1304 cu ft (77%) | Weight: 6925 lbs (99%)
  Day 9: ['CUST4656', 'CUST7884', 'CUST5141']
         Volume: 573 cu ft (34%) | Weight: 2655 lbs (38%)


## Section 10 — Objective Function & Constraints

### Objective Function

Minimize total operational cost **Z** across all delivery days:

$$Z = \sum_{d} \left[ \text{Labor}(d) + \text{Truck}(d) + \text{Holding}(d) \right]$$

| Component | Formula | Source |
|-----------|---------|--------|
| Labor Cost | (2 × $25 + 1 × $30) × paid\_hours × 1.30 | RH case — Costs and Logistics |
| Truck Cost | total\_miles × $1.205/mile | Exhibit III — Henry 2020 |
| Holding Cost | Σ (small items × $5 + large items × $10) for undelivered customers | RH case — Costs and Logistics |

---

### Constraints

| # | Constraint | Value | Description |
|---|-----------|-------|-------------|
| C1 | Σ Vol(i) ≤ 1,700 cu ft | per day | Truck volume capacity |
| C2 | Σ Wt(i) ≤ 7,000 lbs | per day | Truck weight capacity |
| C3 | total\_time(d) ≤ 540 min | per day | 8:00 AM – 5:00 PM window |
| C4 | Every customer served exactly once | — | All 19 orders fulfilled |
| C5 | Truck departs from and returns to DC | each day | Depot constraint |

---

### CO₂ Tracking (not part of objective)

$$\text{CO}_2(d) = \text{miles}(d) \times 1.69 \ \frac{\text{kg}}{\text{mile}} + \text{idle\_hours}(d) \times 3.97 \ \frac{\text{kg}}{\text{hr}}$$

Source: U.S. EPA — medium-duty freight vehicles

---

### Implementation

1. **Section 9 — Grouping:** Greedy packer minimizes delivery days → minimizes labor and holding costs
2. **Section 10 — Routing:** OR-Tools TSP minimizes driving distance → minimizes truck operating cost

In [ ]:
def optimize_route(day_customers):
    """Return optimal stop order for a list of customer IDs."""
    if len(day_customers) <= 1:
        return day_customers
    if ORTOOLS_AVAILABLE:
        return _ortools_route(day_customers)
    else:
        return _nearest_neighbour(day_customers)


def _ortools_route(day_customers):
    nodes = [DC_ID] + day_customers
    n     = len(nodes)
    dist  = [[int(DIST_MATRIX[NODE_INDEX[nodes[i]]][NODE_INDEX[nodes[j]]] * TRUCK_COST_MID * 100)
              for j in range(n)] for i in range(n)]

    manager = pywrapcp.RoutingIndexManager(n, 1, 0)
    routing = pywrapcp.RoutingModel(manager)
    cb_idx  = routing.RegisterTransitCallback(
        lambda f, t: dist[manager.IndexToNode(f)][manager.IndexToNode(t)]
    )
    routing.SetArcCostEvaluatorOfAllVehicles(cb_idx)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.time_limit.seconds = 10

    solution = routing.SolveWithParameters(params)
    if not solution:
        return _nearest_neighbour(day_customers)

    route, idx = [], routing.Start(0)
    while not routing.IsEnd(idx):
        node = manager.IndexToNode(idx)
        if node != 0:
            route.append(nodes[node])
        idx = solution.Value(routing.NextVar(idx))
    return route


def _nearest_neighbour(day_customers):
    unvisited, route, current = list(day_customers), [], DC_ID
    while unvisited:
        nearest = min(unvisited, key=lambda c: DIST_MATRIX[NODE_INDEX[current]][NODE_INDEX[c]])
        route.append(nearest)
        current = nearest
        unvisited.remove(nearest)
    return route


method = 'OR-Tools' if ORTOOLS_AVAILABLE else 'Nearest-Neighbour'
print(f"Route optimization method: {method}\n")
for i, day in enumerate(all_days, 1):
    route  = optimize_route(day)
    nodes  = [DC_ID] + route + [DC_ID]
    miles  = sum(DIST_MATRIX[NODE_INDEX[nodes[k]]][NODE_INDEX[nodes[k+1]]]
                 for k in range(len(nodes)-1))
    print(f"  Day {i}: DC → {' → '.join(route)} → DC  ({miles:.1f} miles)")

Route optimization method: OR-Tools

  Day 1: DC → CUST2394 → DC  (30.1 miles)
  Day 2: DC → CUST4775 → DC  (39.8 miles)
  Day 3: DC → CUST9328 → DC  (22.0 miles)
  Day 4: DC → CUST6386 → CUST1202 → DC  (29.1 miles)
  Day 5: DC → CUST7841 → CUST8401 → DC  (7.2 miles)
  Day 6: DC → CUST5146 → CUST7287 → DC  (47.6 miles)
  Day 7: DC → CUST7594 → CUST3601 → CUST5881 → DC  (43.4 miles)
  Day 8: DC → CUST6840 → CUST6463 → CUST8298 → CUST7291 → DC  (43.8 miles)
  Day 9: DC → CUST4656 → CUST5141 → CUST7884 → DC  (53.6 miles)


## Section 11 — Building the Full Schedule and Manifest

For each day and each stop the schedule records:
- Arrival and departure times (8:00 AM + 23 min check-in + morning load time)
- Costs: truck operating, labor (with RH 15-min pay rounding), holding
- CO₂: driving (1.69 kg/mile) + idling (3.97 kg/hr)

The manifest records items in **LIFO order** — last delivery stop loaded first (deepest in truck).

**Naive baseline definition:** Same daily groupings visited in alphabetical (unoptimized) stop order.  
This isolates the effect of OR-Tools route optimization specifically.

In [ ]:
def mins_to_time(mins):
    """Convert minutes from 8:00 AM to HH:MM AM/PM string."""
    total  = int(8 * 60 + mins)
    h, m   = divmod(total, 60)
    period = 'AM' if h < 12 else 'PM'
    if h > 12: h -= 12
    if h == 0: h  = 12
    return f"{h}:{m:02d} {period}"


def round_labor_hours(raw_hours):
    """RH pay rounding: 15-min intervals, round up at 7-min mark."""
    quarters = int(raw_hours / PAY_INTERVAL_HRS)
    if (raw_hours % PAY_INTERVAL_HRS) * 60 >= PAY_THRESHOLD_MIN:
        quarters += 1
    return quarters * PAY_INTERVAL_HRS


def calc_naive_baseline(all_days):
    """Naive CO₂: same groups, alphabetical stop order (unoptimized routing)."""
    total_miles = 0.0
    total_co2   = 0.0
    for group in all_days:
        naive_route = sorted(group)
        prev = 'DC'
        miles = 0.0
        idle_hrs = 0.0
        for cid in naive_route:
            miles    += DIST_MATRIX[NODE_INDEX[prev]][NODE_INDEX[cid]]
            idle_hrs += cust_dict[cid]['Service_Time_Min'] / 60
            prev      = cid
        miles      += DIST_MATRIX[NODE_INDEX[prev]][0]
        total_miles += miles
        total_co2   += miles * CO2_PER_MILE + idle_hrs * CO2_PER_HOUR
    return round(total_miles, 1), round(total_co2, 1)


NAIVE_MILES, NAIVE_CO2 = calc_naive_baseline(all_days)
print(f"Naive baseline — Miles: {NAIVE_MILES:.1f} | CO₂: {NAIVE_CO2:.1f} kg")

Naive baseline — Miles: 334.9 | CO₂: 661.6 kg


In [ ]:
# ── Build order lines lookup for manifest ──────────────────────────────────────
order_lines = {}
for cid, grp in df_merged.groupby('Customer_ID'):
    order_lines[cid] = grp[['Brand','Product_Description','Quantity',
                              'Vol_CuFt','Weight_Lbs','Size_Class']].to_dict('records')

schedule_rows = []
manifest_rows = []
delivered     = set()
cum_cost = cum_co2 = 0.0

for day_num, day_custs in enumerate(all_days, 1):
    route         = optimize_route(day_custs)
    day_load_time = sum(cust_dict[c]['Load_Time_Min'] for c in route)
    current_time  = CHECKIN_MIN + day_load_time
    prev_node     = DC_ID
    day_miles     = 0.0
    stop_details  = []

    for stop_num, cid in enumerate(route, 1):
        c           = cust_dict[cid]
        ni          = NODE_INDEX[cid]
        transit_min = TIME_MATRIX[NODE_INDEX[prev_node]][ni]
        miles       = DIST_MATRIX[NODE_INDEX[prev_node]][ni]
        arrival     = current_time + transit_min
        departure   = arrival + c['Service_Time_Min']
        current_time = departure
        day_miles   += miles
        prev_node    = cid
        stop_details.append({
            'stop_num': stop_num, 'cid': cid,
            'arrival': arrival, 'departure': departure,
            'transit_min': round(transit_min, 1),
            'service_min': round(c['Service_Time_Min'], 1),
            'miles': round(miles, 2),
        })

    day_miles   += DIST_MATRIX[NODE_INDEX[prev_node]][0]
    return_time  = current_time + TIME_MATRIX[NODE_INDEX[prev_node]][0]
    raw_hrs      = return_time / 60
    paid_hrs     = round_labor_hours(raw_hrs)

    labor_cost   = (2*CREW_BASE_WAGE + DRIVER_WAGE) * paid_hrs * BENEFITS_MULTIPLIER
    truck_cost   = day_miles * TRUCK_COST_MID
    holding_cost = sum(
        cust_dict[c]['Small_Items'] * HOLDING_SMALL +
        cust_dict[c]['Large_Items'] * HOLDING_LARGE
        for c in cust_dict if c not in delivered and c not in set(route)
    ) if day_num > 1 else 0
    day_cost     = labor_cost + truck_cost + holding_cost

    idle_hrs  = sum(s['service_min'] for s in stop_details) / 60
    co2_drive = day_miles * CO2_PER_MILE
    co2_idle  = idle_hrs  * CO2_PER_HOUR
    day_co2   = co2_drive + co2_idle

    cum_cost += day_cost
    cum_co2  += day_co2

    for s in stop_details:
        cal_date, day_name = day_num_to_calendar(day_num)
        schedule_rows.append({
            'Day':               day_num,
            'Day_Name':          day_name,
            'Date':              cal_date,
            'Truck':             'Truck 1',
            'Crew':              'Crew A',
            'Stop_No':           s['stop_num'],
            'Customer_ID':       s['cid'],
            'Address':           cust_dict[s['cid']]['Address'],
            'Window_Start':      '8:00 AM',
            'Window_End':        '5:00 PM',
            'Delivery_Window':   f"{mins_to_time(s['arrival'])} – {mins_to_time(s['departure'])}",
            'Arrival_Time':      mins_to_time(s['arrival']),
            'Departure_Time':    mins_to_time(s['departure']),
            'Transit_Min':       s['transit_min'],
            'Service_Min':       s['service_min'],
            'Miles_From_Prev':   s['miles'],
            'Day_Total_Miles':   round(day_miles, 2),
            'Morning_Load_Min':  round(day_load_time, 1),
            'Return_Time':       mins_to_time(return_time),
            'Raw_Hours':         round(raw_hrs, 2),
            'Paid_Hours':        round(paid_hrs, 2),
            'Labor_Cost':        round(labor_cost, 2),
            'Truck_Oper_Cost':   round(truck_cost, 2),
            'Holding_Cost':      round(holding_cost, 2),
            'Day_Total_Cost':    round(day_cost, 2),
            'Cumulative_Cost':   round(cum_cost, 2),
            'CO2_Drive_kg':      round(co2_drive, 2),
            'CO2_Idle_kg':       round(co2_idle, 2),
            'Day_CO2_kg':        round(day_co2, 2),
            'Cumulative_CO2_kg': round(cum_co2, 2),
            'Naive_CO2_kg':      NAIVE_CO2,
            'CO2_Savings_kg':    round(NAIVE_CO2 - cum_co2, 2),
        })

    for stop_num, cid in enumerate(route, 1):
        load_order = len(route) - stop_num + 1
        for line in order_lines.get(cid, []):
            manifest_rows.append({
                'Day':              day_num,
                'Load_Order':       load_order,
                'Delivery_Stop':    stop_num,
                'Customer_ID':      cid,
                'Brand':            line['Brand'],
                'Product_Desc':     line['Product_Description'],
                'Quantity':         line['Quantity'],
                'Vol_CuFt_Each':    line['Vol_CuFt'],
                'Vol_CuFt_Total':   round(line['Vol_CuFt'] * line['Quantity'], 2),
                'Weight_Lbs_Each':  line['Weight_Lbs'],
                'Weight_Lbs_Total': line['Weight_Lbs'] * line['Quantity'],
                'Size_Class':       line['Size_Class'],
            })

    delivered.update(route)

df_schedule = pd.DataFrame(schedule_rows)
df_manifest = pd.DataFrame(manifest_rows).sort_values(['Day','Load_Order'])

print(f"Schedule built: {len(df_schedule)} stop rows across {df_schedule['Day'].nunique()} days")
print(f"Manifest built: {len(df_manifest)} item rows")

Schedule built: 19 stop rows across 9 days
Manifest built: 168 item rows


## Section 12 — Results Summary

In [ ]:
total_miles = df_schedule.groupby('Day')['Day_Total_Miles'].last().sum()
total_cost  = df_schedule.groupby('Day')['Day_Total_Cost'].last().sum()
total_co2   = df_schedule.groupby('Day')['Day_CO2_kg'].last().sum()

savings_miles = NAIVE_MILES - total_miles
savings_co2   = NAIVE_CO2   - total_co2
savings_pct   = savings_co2 / NAIVE_CO2 * 100 if NAIVE_CO2 > 0 else 0

print("=" * 55)
print("  RH WHITE GLOVE DELIVERY — OPTIMIZATION RESULTS")
print("=" * 55)
print(f"  Total delivery days:       {df_schedule['Day'].nunique()}")
print(f"  Total customer stops:      {len(df_schedule)}")
print(f"  Total miles (optimized):   {total_miles:.1f} miles")
print(f"  Total miles (naive):       {NAIVE_MILES:.1f} miles")
print(f"  Miles saved:               {savings_miles:.1f} miles ({savings_miles/NAIVE_MILES*100:.1f}%)")
print(f"  Total operational cost:    ${total_cost:,.2f}")
print(f"  Total CO₂ (optimized):     {total_co2:.1f} kg")
print(f"  Total CO₂ (naive):         {NAIVE_CO2:.1f} kg")
print(f"  CO₂ savings:               {savings_co2:.1f} kg ({savings_pct:.1f}% reduction)")
print("=" * 55)

  RH WHITE GLOVE DELIVERY — OPTIMIZATION RESULTS
  Total delivery days:       9
  Total customer stops:      19
  Total miles (optimized):   316.6 miles
  Total miles (naive):       334.9 miles
  Miles saved:               18.3 miles (5.5%)
  Total operational cost:    $13,620.52
  Total CO₂ (optimized):     630.7 kg
  Total CO₂ (naive):         661.6 kg
  CO₂ savings:               30.9 kg (4.7% reduction)


In [ ]:
# Daily breakdown
daily = df_schedule.groupby('Day').agg(
    Stops        = ('Stop_No',        'max'),
    Miles        = ('Day_Total_Miles', 'last'),
    Labor_Cost   = ('Labor_Cost',      'last'),
    Truck_Cost   = ('Truck_Oper_Cost', 'last'),
    Holding_Cost = ('Holding_Cost',    'last'),
    Total_Cost   = ('Day_Total_Cost',  'last'),
    CO2_kg       = ('Day_CO2_kg',      'last'),
    Return_Time  = ('Return_Time',     'last'),
).reset_index()
print("Daily breakdown:")
daily

Daily breakdown:


,Day,Stops,Miles,Labor_Cost,Truck_Cost,Holding_Cost,Total_Cost,CO2_kg,Return_Time
0,1,1,30.1,520.0,36.27,0.0,556.27,60.07,12:57 PM
1,2,1,39.8,442.0,47.96,1945.0,2434.96,75.14,12:17 PM
2,3,1,22.0,442.0,26.51,1730.0,2198.51,45.45,12:15 PM
3,4,2,29.1,494.0,35.07,1515.0,2044.07,58.08,12:42 PM
4,5,2,7.2,572.0,8.68,1210.0,1790.68,24.18,1:27 PM
5,6,2,47.6,650.0,57.36,905.0,1612.36,92.45,2:12 PM
6,7,3,43.4,598.0,52.30,650.0,1300.30,85.39,1:49 PM
7,8,4,43.8,754.0,52.78,240.0,1046.78,89.24,3:20 PM
8,9,3,53.6,572.0,64.59,0.0,636.59,100.71,1:32 PM


In [ ]:
# Full schedule view
print("Full delivery schedule:")
df_schedule[['Day','Date','Day_Name','Stop_No','Customer_ID','Address',
             'Arrival_Time','Departure_Time','Delivery_Window',
             'Transit_Min','Service_Min','Miles_From_Prev',
             'Day_Total_Cost','Day_CO2_kg']]

Full delivery schedule:


,Day,Date,Day_Name,Stop_No,Customer_ID,Address,Arrival_Time,Departure_Time,Delivery_Window,Transit_Min,Service_Min,Miles_From_Prev,Day_Total_Cost,Day_CO2_kg
0,1,2026-04-07,Tuesday,1,CUST2394,"1615 La Vereda Rd, Berkeley, CA 94709",10:08 AM,12:27 PM,10:08 AM – 12:27 PM,30,139.0,14.9,556.27,60.07
1,2,2026-04-08,Wednesday,1,CUST4775,13447 Campus Drive – 94619 Oakland,9:51 AM,11:50 AM,9:51 AM – 11:50 AM,27,119.0,19.8,2434.96,75.14
2,3,2026-04-09,Thursday,1,CUST9328,"1298 63rd St, Emeryville, CA 94608",9:50 AM,11:55 AM,9:50 AM – 11:55 AM,20,125.0,11.1,2198.51,45.45
3,4,2026-04-10,Friday,1,CUST6386,"1820 3rd Street, Alameda, CA 94501",9:57 AM,10:16 AM,9:57 AM – 10:16 AM,28,19.0,14.1,2044.07,58.08
4,4,2026-04-10,Friday,2,CUST1202,"765 Eagle Avenue, Alameda, CA 94501",10:20 AM,12:15 PM,10:20 AM – 12:15 PM,4,115.5,1.0,2044.07,58.08
5,5,2026-04-11,Saturday,1,CUST7841,"181 Fremont St UNIT 75A, San Francisco, CA 94105",10:00 AM,11:05 AM,10:00 AM – 11:05 AM,14,65.0,2.9,1790.68,24.18
6,5,2026-04-11,Saturday,2,CUST8401,"1150 Sacramento St UNIT 401, San Francisco, CA...",11:13 AM,1:10 PM,11:13 AM – 1:10 PM,8,116.5,1.1,1790.68,24.18
7,6,2026-04-14,Tuesday,1,CUST5146,"233 Village Way, South San Francisco, CA 94080",10:03 AM,11:33 AM,10:03 AM – 11:33 AM,15,90.5,8.7,1612.36,92.45
8,6,2026-04-14,Tuesday,2,CUST7287,"1515 Park Street, Alameda, CA 94501",12:16 PM,1:47 PM,12:16 PM – 1:47 PM,43,91.0,23.3,1612.36,92.45
9,7,2026-04-15,Wednesday,1,CUST7594,"2046 San Antonio Avenue, Alameda, CA 94501",10:06 AM,11:18 AM,10:06 AM – 11:18 AM,27,72.0,16.2,1300.30,85.39


In [ ]:
# Manifest preview — Day 1
print("Loading manifest (Day 1 — LIFO: Load_Order 1 = loaded first = deepest in truck):")
df_manifest[df_manifest['Day'] == 1]

Loading manifest (Day 1 — LIFO: Load_Order 1 = loaded first = deepest in truck):


,Day,Load_Order,Delivery_Stop,Customer_ID,Brand,Product_Desc,Quantity,Vol_CuFt_Each,Vol_CuFt_Total,Weight_Lbs_Each,Weight_Lbs_Total,Size_Class
0,1,1,1,CUST2394,Reclaimed Oak Reeded Round Dining Table,48'' Table,1,31.42,31.42,306,306,large
1,1,1,1,CUST2394,Nicola Track Vegan Leather Dining Side Chair,Dining Chair,4,9.25,37.00,16,64,small
2,1,1,1,CUST2394,Byron Emperador Dining Table,96'' Dining Table,1,66.67,66.67,484,484,large
3,1,1,1,CUST2394,Arno Fabric Dining Armchair,Dining Chair,8,8.45,67.60,17,136,small
4,1,1,1,CUST2394,Aero Wood Round Dining Table,30'' Bistro Table,1,12.27,12.27,31,31,large
5,1,1,1,CUST2394,Nicola Track Vegan Leather Dining Side Chair,Dining Chair,4,9.25,37.00,16,64,small
6,1,1,1,CUST2394,Arno Vegan Leather Dining Side Chair,Dining Side Chair,2,7.87,15.74,15,30,small
7,1,1,1,CUST2394,Ligné Panel Bed with Footboard,Queen Bed,1,180.81,180.81,398,398,large
8,1,1,1,CUST2394,Byron Canopy Bed,King Bed 102'',1,416.26,416.26,510,510,large
9,1,1,1,CUST2394,Melbourne Bar Cabinet,Bar Cabinet,1,17.50,17.50,211,211,large


## Section 13 — Exporting to Excel for Power BI

Exports four sheets to `RH_schedule_output.xlsx`:
- **Schedule** — one row per stop (stop-level detail for maps and timing)
- **Manifest** — one row per item in LIFO load order
- **Daily_Summary** — one row per day (use for all cost/CO₂ cards and charts — no double counting)
- **Assumptions** — all model parameters with sources

Upload this file to SharePoint and connect Power BI to it.

In [ ]:
# ── Build assumptions table ────────────────────────────────────────────────────
assumptions = pd.DataFrame([
    ('Warehouse address',             DC_ADDRESS,                                          '—',              'RH case document'),
    ('Truck capacity — volume',       TRUCK_CAPACITY_CUFT,                                 'cu ft',          'RH case — 26-foot cargo box'),
    ('Truck capacity — weight',       TRUCK_CAPACITY_LBS,                                  'lbs',            'RH case — Costs and Logistics'),
    ('Crew size',                     3,                                                   'persons',        'RH case — 1 driver + 2 crew'),
    ('Crew base wage',                f'${CREW_BASE_WAGE:.2f}',                             '$/hr',           'RH case — Costs and Logistics'),
    ('Driver wage',                   f'${DRIVER_WAGE:.2f}',                               '$/hr',           'RH case — 20% premium'),
    ('Benefits multiplier',           BENEFITS_MULTIPLIER,                                  'x on wages',     'RH case — health + payroll taxes'),
    ('Pay rounding interval',         '15 minutes',                                         'min',            'RH case — Costs and Logistics'),
    ('Pay rounding threshold',        '7 minutes',                                          'min',            'RH case — rounds up at 7 min'),
    ('Truck cost — low',              f'${TRUCK_COST_LOW:.2f}',                             '$/mile',         'Exhibit III — Henry 2020'),
    ('Truck cost — high',             f'${TRUCK_COST_HIGH:.2f}',                            '$/mile',         'Exhibit III — Henry 2020'),
    ('Truck cost — used (midpoint)',  f'${TRUCK_COST_MID:.3f}',                             '$/mile',         'Exhibit III — Henry 2020'),
    ('Holding cost — small items',    f'${HOLDING_SMALL:.2f}',                              '$/item/day',     'RH case — Costs and Logistics'),
    ('Holding cost — large items',    f'${HOLDING_LARGE:.2f}',                              '$/item/day',     'RH case — Costs and Logistics'),
    ('Start-of-day check-in',         CHECKIN_MIN,                                          'min (once/day)', 'RH case Exhibit V'),
    ('Arrival coordination/stop',     COORD_PER_STOP_MIN,                                   'min/stop',       'RH case Exhibit V'),
    ('Sign-off time/stop',            SIGNOFF_PER_STOP_MIN,                                 'min/stop',       'RH case Exhibit V'),
    ('Delivery window',               '8:00 AM – 5:00 PM',                                  'Tu–Sa',          'RH case — Costs and Logistics'),
    ('Flat rate delivery (local)',    '$299',                                                '$/order',        'Exhibit II — RH Unlimited Furniture Delivery'),
    ('CO₂ — driving',                CO2_PER_MILE,                                          'kg CO₂/mile',    'U.S. EPA medium-duty freight'),
    ('CO₂ — idling',                 CO2_PER_HOUR,                                          'kg CO₂/hr',      'EPA diesel idle emission factor'),
    ('Naive baseline definition',     'Same daily groups, alphabetical stop order',          '—',              'Isolates OR-Tools routing benefit'),
    ('Product volumes & weights',     'rh.com product pages',                               'cu ft / lbs',    'Furniture Details file'),
    ('Loading/unloading times',       'Per item type',                                       'min/item',       'RH case Exhibit V'),
    ('Customer addresses',            'RH customer database',                               '—',              'Customers Addresses file'),
    ('Distance matrix',               'Google Maps Distance Matrix API',                     'miles / min',    'Pulled April 2026 — hardcoded in notebook'),
], columns=['Parameter', 'Value', 'Unit', 'Source'])

In [ ]:
# ── Build daily summary (one row per day — no double counting in Power BI) ────
daily_summary = df_schedule.groupby('Day').agg(
    Date              = ('Date',              'first'),
    Day_Name          = ('Day_Name',          'first'),
    Truck             = ('Truck',             'first'),
    Crew              = ('Crew',              'first'),
    Stops             = ('Stop_No',           'max'),
    Day_Total_Miles   = ('Day_Total_Miles',   'first'),
    Morning_Load_Min  = ('Morning_Load_Min',  'first'),
    Return_Time       = ('Return_Time',       'first'),
    Raw_Hours         = ('Raw_Hours',         'first'),
    Paid_Hours        = ('Paid_Hours',        'first'),
    Labor_Cost        = ('Labor_Cost',        'first'),
    Truck_Oper_Cost   = ('Truck_Oper_Cost',   'first'),
    Holding_Cost      = ('Holding_Cost',      'first'),
    Day_Total_Cost    = ('Day_Total_Cost',    'first'),
    Cumulative_Cost   = ('Cumulative_Cost',   'last'),
    CO2_Drive_kg      = ('CO2_Drive_kg',      'first'),
    CO2_Idle_kg       = ('CO2_Idle_kg',       'first'),
    Day_CO2_kg        = ('Day_CO2_kg',        'first'),
    Cumulative_CO2_kg = ('Cumulative_CO2_kg', 'last'),
    Naive_CO2_kg      = ('Naive_CO2_kg',      'first'),
    CO2_Savings_kg    = ('CO2_Savings_kg',    'last'),
).reset_index()

print(f"Daily summary: {len(daily_summary)} rows (one per day)")
daily_summary

Daily summary: 9 rows (one per day)


,Day,Date,Day_Name,Truck,Crew,Stops,Day_Total_Miles,Morning_Load_Min,Return_Time,Raw_Hours,...,Truck_Oper_Cost,Holding_Cost,Day_Total_Cost,Cumulative_Cost,CO2_Drive_kg,CO2_Idle_kg,Day_CO2_kg,Cumulative_CO2_kg,Naive_CO2_kg,CO2_Savings_kg
0,1,2026-04-07,Tuesday,Truck 1,Crew A,1,30.1,75.5,12:57 PM,4.96,...,36.27,0.0,556.27,556.27,50.87,9.20,60.07,60.07,661.6,601.53
1,2,2026-04-08,Wednesday,Truck 1,Crew A,1,39.8,61.5,12:17 PM,4.29,...,47.96,1945.0,2434.96,2991.23,67.26,7.87,75.14,135.20,661.6,526.40
2,3,2026-04-09,Thursday,Truck 1,Crew A,1,22.0,67.0,12:15 PM,4.25,...,26.51,1730.0,2198.51,5189.74,37.18,8.27,45.45,180.65,661.6,480.95
3,4,2026-04-10,Friday,Truck 1,Crew A,2,29.1,66.0,12:42 PM,4.71,...,35.07,1515.0,2044.07,7233.81,49.18,8.90,58.08,238.73,661.6,422.87
4,5,2026-04-11,Saturday,Truck 1,Crew A,2,7.2,83.5,1:27 PM,5.45,...,8.68,1210.0,1790.68,9024.48,12.17,12.01,24.18,262.91,661.6,398.69
5,6,2026-04-14,Tuesday,Truck 1,Crew A,2,47.6,85.0,2:12 PM,6.21,...,57.36,905.0,1612.36,10636.84,80.44,12.01,92.45,355.36,661.6,306.24
6,7,2026-04-15,Wednesday,Truck 1,Crew A,3,43.4,76.0,1:49 PM,5.82,...,52.30,650.0,1300.30,11937.14,73.35,12.04,85.39,440.75,661.6,220.85
7,8,2026-04-16,Thursday,Truck 1,Crew A,4,43.8,94.5,3:20 PM,7.34,...,52.78,240.0,1046.78,12983.92,74.02,15.22,89.24,529.99,661.6,131.61
8,9,2026-04-17,Friday,Truck 1,Crew A,3,53.6,57.0,1:32 PM,5.53,...,64.59,0.0,636.59,13620.50,90.58,10.12,100.71,630.70,661.6,30.90


In [ ]:
# ── Write to Excel ─────────────────────────────────────────────────────────────
output_file = 'RH_schedule_output.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_schedule.to_excel(writer,   sheet_name='Schedule',      index=False)
    df_manifest.to_excel(writer,   sheet_name='Manifest',      index=False)
    daily_summary.to_excel(writer, sheet_name='Daily_Summary', index=False)
    assumptions.to_excel(writer,   sheet_name='Assumptions',   index=False)

print(f"Exported: {output_file}")
print(f"  Schedule:      {len(df_schedule)} rows")
print(f"  Manifest:      {len(df_manifest)} rows")
print(f"  Daily_Summary: {len(daily_summary)} rows")
print(f"  Assumptions:   {len(assumptions)} rows")

Exported: RH_schedule_output.xlsx
  Schedule:      19 rows
  Manifest:      168 rows
  Daily_Summary: 9 rows
  Assumptions:   26 rows


In [ ]:
# ── Download ───────────────────────────────────────────────────────────────────
files.download('RH_schedule_output.xlsx')
print("Download started.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


---

## Disruption Scenario A — Customer Not Available

To simulate a disruption, run the cell below to override `all_days`,  
then re-run from **Section 10** onwards and re-export.

**Scenario:** After Stop 1 on Day 8, CUST8298 is not available.  
Remove from Day 8 and reschedule to Day 9.

In [ ]:
# ── Uncomment to apply disruption ─────────────────────────────────────────────
# all_days[7] = ['CUST6840', 'CUST7291', 'CUST6463']          # Day 8 — skip CUST8298
# all_days[8] = ['CUST4656', 'CUST7884', 'CUST5141', 'CUST8298']  # Day 9 — add CUST8298
# print('Disruption applied: CUST8298 moved from Day 8 to Day 9')
# print(f'Day 8: {all_days[7]}')
# print(f'Day 9: {all_days[8]}')
# # Then re-run from Section 10 (route optimization) through Section 13 (export)
print("Disruption cell ready — uncomment lines above to apply.")

Disruption cell ready — uncomment lines above to apply.
